### IMPORT

In [ ]:
from pathlib import Path

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader
import rootutils

rootutils.setup_root(__file__, indicator=".project-root", pythonpath=True)

from src.utils.bev_emb import BEVAdapter
from src.utils.bevqa import BEVQA
from src.utils.bevqa_dataset import BEVQADataset
from src.utils.head import OutputHead
from src.utils.mca import MCALayer
from src.utils.text_emb import TextAdapter
from src.utils.train import train_epoch, val_epoch

device = "cuda" if torch.cuda.is_available() else "cpu"

### PATH

In [ ]:
ROOT = Path("/home/robesafe-sandra/BEV")

MINI_DIR = ROOT / "data/dataset_mini"
MINI_DICT_DIR = MINI_DIR / "dict_mini"
MINI_FEAT_DIR = MINI_DIR / "bev_features_mini"

GLOVE = ROOT/ "glove.6B/glove.6B.300d.txt"
#hydra 

### DATASET

In [3]:
train_dataset = BEVQADataset(
    bev_dir=MINI_FEAT_DIR / "train_mini",
    json_path=MINI_DICT_DIR / "NuScenes_train_questions_mini.json",
    glove=GLOVE
)

In [4]:
val_dataset = BEVQADataset(
    bev_dir=MINI_FEAT_DIR / "val_mini",
    json_path=MINI_DICT_DIR / "NuScenes_val_questions_mini.json",
    glove=GLOVE
)

In [5]:
feat, quest, ans = train_dataset[0]
print(feat.shape, quest.shape, ans)

torch.Size([128, 200, 200]) torch.Size([35, 300]) tensor(26)


### DATALOADER

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [7]:
feat, quest, ans = next(iter(train_dataloader))
print(feat.shape, quest.shape, ans)

torch.Size([4, 128, 200, 200]) torch.Size([4, 35, 300]) tensor([29, 23, 18, 23])


### EMBEDDINGS

In [8]:
bev_ad = BEVAdapter() # BEV MAP -> BEV EMB
text_ad = TextAdapter() # [B,35,512] to match BEV emb [B,400,512] 
feat_output = bev_ad(feat)
text_output = text_ad(quest)
print(f"Feat:{feat_output.shape}") # [B,400,512]
print(f"Text:{text_output.shape}") # [B,35,512]

Feat:torch.Size([4, 400, 512])
Text:torch.Size([4, 35, 512])


### MCAN

In [9]:
mca = MCALayer()
bev_out, text_out = mca(feat_output, text_output)
print(bev_out.shape, text_out.shape) # [4, 400, 512], [4, 35, 512]

torch.Size([4, 400, 512]) torch.Size([4, 35, 512])


### HEAD

In [10]:
head = OutputHead()
out = head(text_out)
print(out.shape) # [4,30]

torch.Size([4, 30])


### MODEL

In [11]:
model = BEVQA()
out = model(feat, quest)
print(out.shape) # [4, 30]

torch.Size([4, 30])


### TRAIN

In [13]:
model = BEVQA().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

num_epochs = 10

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_epoch(model, train_dataloader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_dataloader, criterion, device)
    print(f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
              f"Train Loss: {tr_loss:.4f} - Acc: {tr_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} - Acc: {val_acc:.4f}")

Epoch 01/10 | Train Loss: 1.5805 - Acc: 0.4251 | Val Loss: 6.1615 - Acc: 0.0396
Epoch 02/10 | Train Loss: 1.1702 - Acc: 0.4969 | Val Loss: 6.6585 - Acc: 0.0663
Epoch 03/10 | Train Loss: 1.0702 - Acc: 0.5449 | Val Loss: 6.7818 - Acc: 0.0645
Epoch 04/10 | Train Loss: 1.0053 - Acc: 0.5752 | Val Loss: 6.9250 - Acc: 0.0635
Epoch 05/10 | Train Loss: 0.9575 - Acc: 0.5896 | Val Loss: 7.3950 - Acc: 0.0626
Epoch 06/10 | Train Loss: 0.9110 - Acc: 0.6095 | Val Loss: 7.7513 - Acc: 0.0534
Epoch 07/10 | Train Loss: 0.8498 - Acc: 0.6383 | Val Loss: 7.9855 - Acc: 0.0709
Epoch 08/10 | Train Loss: 0.7971 - Acc: 0.6576 | Val Loss: 8.3612 - Acc: 0.0433
Epoch 09/10 | Train Loss: 0.7538 - Acc: 0.6700 | Val Loss: 8.3423 - Acc: 0.0442
Epoch 10/10 | Train Loss: 0.7083 - Acc: 0.6961 | Val Loss: 8.6441 - Acc: 0.0506
